In [ ]:
import jax
# Set before creating JAX arrays to preserve float64 precision.
jax.config.update("jax_enable_x64", True)

import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
import os
import sys

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from src.config import ExperimentConfig
from src.control.pulses import apply_low_pass_filter
from src.calibration.training_loops import make_setup, make_cz_optimizer, train_pulse
from src.analysis.metrics import cz_process_fidelity

config = ExperimentConfig.load("configs/two_qubit_CZ_zz.yaml")
cal_g, cal_d, cal_a = np.load(config.artifact_path("cal_params.npy"))


In [ ]:
N_HARMONICS = 8
N_STEPS = 500

H0, Hz, bound_loss_fn, init_params, basis, kernel, f_max = make_setup(
    config,
    n_harmonics=N_HARMONICS,
    coupling_mhz=cal_g,
    detuning_mhz=cal_d,
    anharmonicity_mhz=cal_a,
)
optimizer = make_cz_optimizer(0.01, N_STEPS)

print("Optimizing Two-Qubit CZ Gate...")
opt_params, history = train_pulse(bound_loss_fn, init_params, optimizer, N_STEPS)


In [ ]:
# --- Plotting ---
tlist = np.arange(0, config.pulse.duration_ns, config.pulse.dt_ns)

# Reconstruct the flux pulse and convert back to GHz
flux_rad = f_max * jnp.tanh(basis @ opt_params['flux_coeffs'] / f_max)
final_flux_ghz = apply_low_pass_filter(flux_rad, kernel) / (2 * jnp.pi)
saved_pulse_ghz = np.load(config.artifact_path("cal_pulse_rad.npy")) / (2 * jnp.pi)
print(f"Maximum difference from pulse saved by 01: {np.max(np.abs(final_flux_ghz - saved_pulse_ghz)):.3e} GHz")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history['loss'], label='Total Loss', color='black')
ax1.plot(history['infidelity'], label='Infidelity', linestyle='--')
ax1.plot(history['leakage'], label='Leakage', linestyle='-.')
ax1.set_yscale('log')
ax1.set_xlabel('Optimization Step')
ax1.set_ylabel('Metric Value')
ax1.legend()
ax1.set_title('CZ Gate Training History')
ax1.grid(True)

ax2.plot(tlist, final_flux_ghz, color='purple', label='Pulse trained in 02')
ax2.plot(tlist, saved_pulse_ghz, color='black', linestyle='--', label='Pulse saved by 01')
ax2.set_xlabel('Time (ns)')
ax2.set_ylabel('Flux Excursion (GHz)')
ax2.set_title('Bandwidth-Limited Flux Control')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()


In [ ]:
# --- Extracting the Unitary Evolution History ---
@jax.jit
def get_unitary_history(params, dt, H0, Hz_control, basis, kernel, f_max):
    """Replays the optimized pulse and saves the matrix at every time step."""
    raw = basis @ params['flux_coeffs']
    flux = f_max * jnp.tanh(raw / f_max)
    flux = apply_low_pass_filter(flux, kernel)

    def step(U, flux_val):
        H = H0 + flux_val * Hz_control
        U_next = jax.scipy.linalg.expm(-1j * H * dt) @ U
        return U_next, U_next 
    
    # 9x9 Identity matrix as the starting state
    U_init = jnp.eye(H0.shape[0], dtype=jnp.complex128)
    _, U_history = jax.lax.scan(step, U_init, flux)
    return U_history

U_hist = get_unitary_history(opt_params, config.pulse.dt_ns, H0, Hz, basis, kernel, f_max)

# --- Calculating Populations and Phase ---
# In our 3x3 Kronecker product: |00>=0, |01>=1, |02>=2, |10>=3, |11>=4, |12>=5, |20>=6
# When starting in |11> (index 4), track population of |11> and the leakage state |20> (index 6)
pop_11 = jnp.abs(U_hist[:, 4, 4])**2
pop_20 = jnp.abs(U_hist[:, 6, 4])**2

# Isolate the conditional phase (in radians, unwrapped to prevent jumping at pi/-pi)
phase_cond_rad = jnp.unwrap(
    jnp.angle(U_hist[:, 4, 4]) - jnp.angle(U_hist[:, 1, 1]) - 
    jnp.angle(U_hist[:, 3, 3]) + jnp.angle(U_hist[:, 0, 0])
)
# Convert to units of Pi
phase_cond_pi = phase_cond_rad / jnp.pi

# --- Plotting ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# The Avoided Crossing Collision
ax1.plot(tlist, pop_11, label='$|11\\rangle$ Population', color='blue', linewidth=2)
ax1.plot(tlist, pop_20, label='$|20\\rangle$ Leakage State', color='red', linestyle='--')
ax1.axhline(1.0, color='gray', linestyle=':', alpha=0.5)
ax1.set_xlabel('Time (ns)')
ax1.set_ylabel('Probability')
ax1.set_title('State Population Dynamics')
ax1.legend()
ax1.grid(True)

# Conditional Phase Accumulation
ax2.plot(tlist, phase_cond_pi, label='Conditional Phase', color='green', linewidth=2)
ax2.axhline(-1.0, color='gray', linestyle=':', label='Target (-1.0 $\pi$)')
ax2.set_xlabel('Time (ns)')
ax2.set_ylabel('Phase Accumulation (Units of $\pi$)')
ax2.set_title('180° ($\pi$) Phase Shift Evolution')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
idx = np.array([0, 1, 3, 4])
U_final = np.asarray(U_hist[-1])                      # final 9x9 unitary from the cell above
res = cz_process_fidelity(U_final[np.ix_(idx, idx)])
print(f"Twin: F_avg = {res['F_avg']*100:.3f}%  F_pro = {res['F_pro']*100:.3f}%  leakage = {res['leakage']*100:.3f}%")